<div style="border-left: 5px solid #b7791f; background-color: #fff8e1; padding: 0.8em 1em; margin: 1em 0; border-radius: 4px;">
  <strong>Warning: AI-assisted materials</strong><br><br>
  These materials were developed with assistance from AI tools. All content has been reviewed and edited by the instructor, who takes final responsibility for its accuracy, clarity, and appropriateness for the course. Students should treat these materials as instructor-reviewed course content while applying the same critical judgment they would use with any technical material. Please report any suspected errors or unclear explanations to ghunt@wm.edu.
</div>

# Model Evaluation and Selection

- **Evaluation:** How well does a model or fitting procedure perform on new data?
- **Selection:** Which model or fitting procedure should we use?

Evaluation gives performance estimates. We may then use those estimates for selection. Once an estimate guides a choice, however, it has become part of the model-building process.


## Performance Metrics

Let our data be
$$
\mathcal D=\{(x_i,y_i)\}_{i=1}^N.
$$

A **performance metric** takes a dataset and a model and returns a number. Write the chosen metric as $M(\mathcal D,f)$. For example:

| Metric | What it measures | Better |
|---|---|---|
| Mean squared error (MSE) | Squared prediction errors | Lower |
| Classification error | Fraction of incorrect labels | Lower |
| Log loss / cross-entropy | Quality of predicted probabilities | Lower |
| F1 score | A combination of precision and recall | Higher |
| Correlation | Linear association between outcomes and predictions | Higher, toward 1 |

For regression,
$$
\operatorname{MSE}(\mathcal D,f)
=\frac1N\sum_{i=1}^N\big(y_i-f(x_i)\big)^2.
$$

MSE, classification error, and log loss average observation-level losses. Other metrics do not need to have that form. F1 is computed from a confusion matrix, and correlation summarizes all outcomes and predictions together.

The evaluation metric also need not equal the loss used to fit the model. We might fit logistic regression using cross-entropy, then evaluate classification error or F1. The metric should match the practical prediction goal.


## What Do We Want to Measure?

Suppose a fitting procedure $A$ uses training data to produce
$$
\hat f=A(\mathcal D_{\mathrm{train}}).
$$

We want to know how $\hat f$ performs on **new observations from the population where it will be used**.

### Holdout evaluation

One practical approach is:

1. Split the available data into $\mathcal D_{\mathrm{train}}$ and $\mathcal D_{\mathrm{eval}}$ (for example, assign $p\%$ to training and $(1-p)\%$ to evaluation).
2. Fit $\hat f=A(\mathcal D_{\mathrm{train}})$.
3. Compute $M(\mathcal D_{\mathrm{eval}},\hat f)$.

For i.i.d. data, a random split gives an evaluation sample that did not influence the model. Its score estimates performance on new data, although a different split would give a different estimate.


## Why Not Evaluate on the Training Data?

Training data are informative about **fit to those observations**. They are not neutral evidence about performance on new observations because we used them to choose the fitted model:
$$
\hat f=A(\mathcal D_{\mathrm{train}}).
$$

The fitting algorithm searched for a function that did well on this particular sample. Any accidental pattern that helps on the sample is rewarded along with real structure.

An analogy: 

- **Practice questions:** studying and grading yourself on the same questions measures how well you learned those questions. A new exam tests whether the learning transfers.

Likewise, low training error may reflect a reproducible relationship, adaptation to random noise, or both. An independent evaluation sample asks which improvements carry over.

Training error still matters. A model with large training error has not even fit the observed data well. The point is that training error alone cannot measure generalization.


**Overfitting** occurs when adaptation to sample-specific variation harms performance on new observations. It usually appears as very low training error together with worse evaluation error.


### Polynomial Regression Example

Generate training and evaluation data independently from
$$
Y=\sin(2\pi X)+\varepsilon,
\qquad X\sim\operatorname{Uniform}(-1,1),
\qquad \varepsilon\sim N(0,0.25^2).
$$

Fit least-squares polynomials of increasing degree. The dashed curve is known only because this is a simulation.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.patches import Patch
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures


def make_sine_data(n, rng, noise_std=0.25):
    x = rng.uniform(-1, 1, size=n)
    y = np.sin(2 * np.pi * x) + rng.normal(0, noise_std, size=n)
    return x, y


def polynomial_model(degree):
    return make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=True),
        LinearRegression(fit_intercept=False),
    )


def fit_polynomial_curves(x_train, y_train, x_eval, y_eval, max_degree=15):
    degrees = np.arange(max_degree + 1)
    models = []
    train_mse = []
    eval_mse = []

    for degree in degrees:
        model = polynomial_model(degree)
        model.fit(x_train.reshape(-1, 1), y_train)
        models.append(model)
        train_mse.append(
            mean_squared_error(y_train, model.predict(x_train.reshape(-1, 1)))
        )
        eval_mse.append(
            mean_squared_error(y_eval, model.predict(x_eval.reshape(-1, 1)))
        )

    return degrees, models, np.array(train_mse), np.array(eval_mse)


rng = np.random.default_rng(455)
x_train, y_train = make_sine_data(30, rng)
x_eval, y_eval = make_sine_data(1000, rng)

degrees, models, train_mse, eval_mse = fit_polynomial_curves(
    x_train, y_train, x_eval, y_eval
)


In [ ]:
#| code-fold: true
grid = np.linspace(-1, 1, 500).reshape(-1, 1)
fig, axes = plt.subplots(1, 4, figsize=(13, 3.5), sharex=True, sharey=True)

for ax, degree in zip(axes, [1, 5, 10, 15]):
    ax.scatter(x_train, y_train, s=20, alpha=0.65, label="Training data")
    ax.plot(grid[:, 0], np.sin(2 * np.pi * grid[:, 0]), "k--", label="True mean")
    ax.plot(grid[:, 0], models[degree].predict(grid), color="tab:red", label="Fit")
    ax.set(title=f"Degree {degree}", xlabel="$x$", ylim=(-2, 2))

axes[0].set_ylabel("$y$")
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()


In [ ]:
#| code-fold: true
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(degrees, np.log10(train_mse), "o-", label="Training MSE")
ax.plot(degrees, np.log10(eval_mse), "o-", label="Evaluation MSE")
ax.set(
    xlabel="Polynomial degree",
    ylabel=r"$\log_{10}(\mathrm{MSE})$",
    xticks=degrees,
    title="One random training/evaluation split",
)
ax.legend()
fig.tight_layout()
plt.show()


A polynomial of degree at most $d+1$ can reproduce any polynomial of degree at most $d$ by setting its last coefficient to zero. Let $\hat f_d$ be the least-squares fit of degree at most $d$. Then
$$
\operatorname{MSE}_{\mathrm{train}}(\hat f_{d+1})
\le
\operatorname{MSE}_{\mathrm{train}}(\hat f_d).
$$

There is no corresponding guarantee for evaluation error. In this run, moderate degrees capture the signal. The highest degrees continue to reduce training error while producing unstable shapes that do not generalize.


### Repeating the Simulation

One split can be unusual. Repeat the entire experiment many times and summarize the error curves across repetitions.


In [ ]:
#| code-fold: true
def repeated_polynomial_experiment(
    n_repetitions=200,
    n_train=30,
    n_eval=1000,
    max_degree=15,
    seed=456,
):
    rng = np.random.default_rng(seed)
    train_curves = []
    eval_curves = []

    for _ in range(n_repetitions):
        x_train, y_train = make_sine_data(n_train, rng)
        x_eval, y_eval = make_sine_data(n_eval, rng)
        degrees, _, train_mse, eval_mse = fit_polynomial_curves(
            x_train, y_train, x_eval, y_eval, max_degree
        )
        train_curves.append(train_mse)
        eval_curves.append(eval_mse)

    return degrees, np.array(train_curves), np.array(eval_curves)


degrees, train_curves, eval_curves = repeated_polynomial_experiment()

fig, ax = plt.subplots(figsize=(8, 4.5))
for curves, label, color in [
    (train_curves, "Training MSE", "tab:blue"),
    (eval_curves, "Evaluation MSE", "tab:orange"),
]:
    log_curves = np.log10(curves)
    median = np.median(log_curves, axis=0)
    q25, q75 = np.percentile(log_curves, [25, 75], axis=0)
    ax.plot(degrees, median, label=label, color=color)
    ax.fill_between(degrees, q25, q75, color=color, alpha=0.18)

ax.set(
    xlabel="Polynomial degree",
    ylabel=r"$\log_{10}(\mathrm{MSE})$",
    xticks=degrees,
    title="Repeated experiments: median and middle 50%",
)
ax.legend()
fig.tight_layout()
plt.show()


The usual cartoon summarizes this common pattern:

![Training and evaluation error versus model complexity.](https://i.sstatic.net/rpqa6.jpg)

The evaluation curve often decreases at first as the model captures more signal, then increases when added flexibility mostly fits sample-specific variation. This U shape is a useful pattern, not a theorem that every dataset must follow.


### Why Can Complexity Increase Overfitting?

Several views reinforce the same point:

- **Capacity:** a richer model class can represent more real structure, but it can also represent more noise.
- **Search:** fitting searches within a collection of functions and chooses one that looks good on the training sample. A larger collection offers more chances to fit accidental patterns.
- **Sensitivity:** highly flexible fits can change substantially when the sample changes slightly. That instability is evidence that some fitted structure may be sample-specific.

Complexity is not inherently bad. A class can be too simple to represent the signal. The goal is enough flexibility to learn structure without relying too heavily on noise, judged using data that did not determine the fit.


## Cross-Validation Estimates Performance

A holdout score depends on one split and fits the model using only part of the data. **Cross-validation (CV)** repeats the fit/evaluate process over several folds.

Start with a fixed fitting procedure $A$, such as least-squares polynomial regression of degree 3. The procedure is fixed even though each training fold produces different fitted coefficients.

### $k$-fold cross-validation

Partition $\mathcal D$ into disjoint folds $\mathcal F_1,\ldots,\mathcal F_k$. For $j=1,\ldots,k$, fit without fold $j$ and evaluate on the omitted fold:
$$
\hat f^{(-j)}=A(\mathcal D\setminus\mathcal F_j),
\qquad
e_j=M(\mathcal F_j,\hat f^{(-j)}).
$$

Return a summary, often
$$
\operatorname{CV}(\mathcal D,A)=\frac1k\sum_{j=1}^k e_j.
$$

Five or ten folds are common. Leave-one-out is the special case $k=N$.

**CV returns a performance estimate for a fixed procedure $A$. It does not, by itself, select a model or hyperparameter.** We may subsequently compare CV estimates to make a selection.


In [ ]:
#| code-fold: true
fig, ax = plt.subplots(figsize=(7, 3))
for row in range(5):
    for fold in range(5):
        color = "tab:orange" if fold == row else "tab:blue"
        ax.barh(row, 1, left=fold, height=0.7, color=color, edgecolor="white")

ax.set(
    xticks=np.arange(5) + 0.5,
    xticklabels=[f"Fold {j}" for j in range(1, 6)],
    yticks=np.arange(5),
    yticklabels=[f"Fit {j}" for j in range(1, 6)],
    xlim=(0, 5),
)
ax.invert_yaxis()
ax.legend(
    handles=[
        Patch(color="tab:blue", label="Train"),
        Patch(color="tab:orange", label="Evaluate"),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=2,
)
fig.tight_layout()
plt.show()


### Example: Estimate the Performance of Degree 3

Generate one dataset and evaluate the fixed procedure $A_3$: fit a degree-3 polynomial by least squares.


In [ ]:
rng = np.random.default_rng(457)
x_cv, y_cv = make_sine_data(100, rng)
cv = KFold(n_splits=5, shuffle=True, random_state=457)

degree3_mse = []
for train_index, eval_index in cv.split(x_cv):
    model = polynomial_model(3)
    model.fit(x_cv[train_index, None], y_cv[train_index])
    predictions = model.predict(x_cv[eval_index, None])
    degree3_mse.append(mean_squared_error(y_cv[eval_index], predictions))

degree3_mse = np.array(degree3_mse)
print("Fold MSEs:", np.round(degree3_mse, 4))
print(f"Mean CV MSE: {degree3_mse.mean():.4f}")


### Which Model Did We Evaluate? None of Them!

Five-fold CV fitted five functions,
$$
\hat f^{(-1)},\ldots,\hat f^{(-5)},
$$
each using a different $4/5$ of the observations. The CV score is not an evaluation of any one of these functions, nor of the final function later fitted on all $N$ observations.

It estimates the performance of the **procedure** $A$ when trained on samples of size approximately $N(k-1)/k$. After evaluation, we ordinarily fit
$$
\hat f_{\mathrm{final}}=A(\mathcal D)
$$
using all available training data. None of the fold models is the final model.

Here $A$ should be construed broadly: everything fixed in advance that turns training data into a fitted predictor. It can include preprocessing, feature construction, an estimator, and fixed settings. If those steps are learned from the data, they must be repeated inside every training fold.

CV uses the available data efficiently, but it does not create independent experiments. The training folds overlap, so the fold scores are generally dependent. Their sample standard deviation is not automatically a standard error for the mean CV score.


## From Evaluation to Model Selection

We have already encountered model selection in the polynomial example. For each degree $d$, define $A_d$ to mean: construct polynomial features through degree $d$ and fit their coefficients by least squares. For example, $A_5$ takes a dataset, constructs the features $1,x,\ldots,x^5$, and fits their coefficients. Comparing degrees $0,\ldots,15$ means comparing the procedures
$$
A_0,A_1,\ldots,A_{15}.
$$

The general notation $A_\lambda$ says the same thing while allowing other choices. The index $\lambda$ might be polynomial degree, number of neighbors, regularization strength, tree depth, or even model family. We write the candidate collection as
$$
\{A_\lambda:\lambda\in\Lambda\}.
$$
For the polynomial example, $\lambda=d$ and $\Lambda=\{0,\ldots,15\}$.

Three objects now matter:

| Object | Meaning |
|---|---|
| $\hat f$ | One fitted prediction function |
| $A_\lambda$ | A fixed fitting procedure for a chosen $\lambda$ |
| $\mathcal P$ | The larger procedure that compares candidates, selects $\lambda$, and refits |

We estimate candidate performance using separate **validation data**. The word *validation* describes the role of these data: their scores guide modeling choices. For any candidate fixed in advance, its validation score can reasonably estimate its performance. Once we choose the candidate with the best score, that score has also influenced the final model.


### The Model-Building Meta-Procedure

Selection creates a larger procedure. Let $\mathcal P$ denote the following **meta-procedure**:

1. Split supplied data into $\mathcal D_{\mathrm{train}}$ and $\mathcal D_{\mathrm{val}}$.
2. For each $\lambda\in\Lambda$, fit and score
   $$
   \hat f_\lambda=A_\lambda(\mathcal D_{\mathrm{train}}),
   \qquad
   e_\lambda=M(\mathcal D_{\mathrm{val}},\hat f_\lambda).
   $$
3. Select $\hat\lambda=\arg\min_{\lambda\in\Lambda}e_\lambda$ (for a lower-is-better metric).
4. Refit $A_{\hat\lambda}$ on all supplied data and return the result.

Symbolically,
$$
\hat f=\mathcal P(\mathcal D).
$$

The final model depends on both the training and validation portions. Evaluation is no longer outside the learning process: inside $\mathcal P$, it helps determine the output.


### Can We Trust the Winning Validation Score?

Validation estimates contain sampling noise. Taking the best score favors candidates whose estimated performance happened to be favorable. Thus $e_{\hat\lambda}$ is usually optimistic as an estimate of the selected model's future performance.

The same capacity ideas now operate at the level of the selection procedure:

- A larger search space $\Lambda$ offers more opportunities to find a favorable validation fluctuation.
- Flexible candidates can differ more substantially from one another.
- Repeatedly checking validation results and revising the search adapts the procedure more strongly to those data.

This is **overfitting to the validation set**. The amount can be small when the search is modest and estimates are stable; it can be serious after a large or adaptive search.

The same warning applies when $e_\lambda$ is a CV estimate. CV first estimates each fixed $A_\lambda$; minimizing those estimates is a separate selection step.


## Train, Validation, and Test

To obtain an independent final evaluation, we can split the data into three sets: 

| Data | Role |
|---|---|
| $\mathcal D_{\mathrm{train}}$ | Fit candidate models |
| $\mathcal D_{\mathrm{val}}$ | Choose models and hyperparameters |
| $\mathcal D_{\mathrm{test}}$ | Evaluate after all choices are finished |

A simple workflow is:

1. Fit candidates on $\mathcal D_{\mathrm{train}}$.
2. Select $\hat\lambda$ using $\mathcal D_{\mathrm{val}}$.
3. Refit $A_{\hat\lambda}$ on $\mathcal D_{\mathrm{train}}\cup\mathcal D_{\mathrm{val}}$.
4. Evaluate once on $\mathcal D_{\mathrm{test}}$.

“Once” means that the test result does not guide another revision. If it does, the test data have become validation data in practice. The names matter less than how the data are used.


## Simulation: Selection and Validation Optimism

We can observe these roles directly in a simulation. Fit polynomial degrees $0,\ldots,20$ on training data, select the smallest validation MSE, and inspect test error only after selection.


In [ ]:
def selection_signal(x):
    return (
        0.6 * np.sin(2 * np.pi * x)
        + 0.45 * np.sin(8 * np.pi * x)
        + 0.3 * np.sin(14 * np.pi * x)
        + np.exp(-120 * (x - 0.2) ** 2)
        - 0.8 * np.exp(-100 * (x + 0.35) ** 2)
    )


def make_selection_data(n, rng, noise_std=1.5):
    x = rng.uniform(-1, 1, size=n)
    y = selection_signal(x) + rng.normal(0, noise_std, size=n)
    return x, y


def run_selection_experiment(
    max_degree=20,
    n_train=250,
    n_val=100,
    n_test=1000,
    noise_std=1.5,
    rng=None,
):
    if rng is None:
        rng = np.random.default_rng()

    x_train, y_train = make_selection_data(n_train, rng, noise_std)
    x_val, y_val = make_selection_data(n_val, rng, noise_std)
    x_test, y_test = make_selection_data(n_test, rng, noise_std)

    degrees = np.arange(max_degree + 1)
    train_mse = []
    val_mse = []
    test_mse = []

    for degree in degrees:
        model = polynomial_model(degree)
        model.fit(x_train[:, None], y_train)
        train_mse.append(mean_squared_error(y_train, model.predict(x_train[:, None])))
        val_mse.append(mean_squared_error(y_val, model.predict(x_val[:, None])))
        test_mse.append(mean_squared_error(y_test, model.predict(x_test[:, None])))

    train_mse = np.array(train_mse)
    val_mse = np.array(val_mse)
    test_mse = np.array(test_mse)
    selected_degree = int(np.argmin(val_mse))

    return {
        "degrees": degrees,
        "train_mse": train_mse,
        "val_mse": val_mse,
        "test_mse": test_mse,
        "selected_degree": selected_degree,
    }


result = run_selection_experiment(rng=np.random.default_rng(458))
d_hat = result["selected_degree"]
print(f"Validation-selected degree: {d_hat}")
print(f"Selected validation MSE: {result['val_mse'][d_hat]:.3f}")
print(f"Test MSE at the selected degree: {result['test_mse'][d_hat]:.3f}")


In [ ]:
#| code-fold: true
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.semilogy(result["degrees"], result["train_mse"], "o-", label="Training MSE")
ax.semilogy(result["degrees"], result["val_mse"], "o-", label="Validation MSE")
ax.semilogy(result["degrees"], result["test_mse"], "o-", label="Test MSE")
ax.axvline(d_hat, color="black", linestyle="--", label=f"Selected degree: {d_hat}")
ax.set(
    xlabel="Polynomial degree",
    ylabel="MSE (log scale)",
    xticks=result["degrees"],
    title="One model-selection experiment",
)
ax.legend()
fig.tight_layout()
plt.show()


The test curve is shown for teaching purposes. In a real analysis, looking at the entire test curve and then reconsidering the degree would use the test set for selection.

The validation and test curves estimate similar performance for each degree, but they are noisy. We specifically select the minimum point of the validation curve, so the selected validation error receives a favorable-selection advantage.


### Monte Carlo View

Repeat the full train/validation/test experiment. The first plot summarizes candidate error curves; the second examines errors at the validation-selected degree.


In [ ]:
#| code-fold: true
rng = np.random.default_rng(459)
n_repetitions = 200
results = [run_selection_experiment(rng=rng) for _ in range(n_repetitions)]

all_train = np.array([r["train_mse"] for r in results])
all_val = np.array([r["val_mse"] for r in results])
all_test = np.array([r["test_mse"] for r in results])
selected_degrees = np.array([r["selected_degree"] for r in results])
degrees = results[0]["degrees"]

fig, ax = plt.subplots(figsize=(9, 4.8))
for curves, label, color in [
    (all_train, "Training MSE", "tab:blue"),
    (all_val, "Validation MSE", "tab:orange"),
    (all_test, "Test MSE", "tab:green"),
]:
    log_curves = np.log10(curves)
    median = np.median(log_curves, axis=0)
    q25, q75 = np.percentile(log_curves, [25, 75], axis=0)
    ax.plot(degrees, median, label=label, color=color)
    ax.fill_between(degrees, q25, q75, color=color, alpha=0.15)

ax.set(
    xlabel="Polynomial degree",
    ylabel=r"$\log_{10}(\mathrm{MSE})$",
    xticks=degrees,
    title="Repeated selection experiments: median and middle 50%",
)
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
#| code-fold: true
selected_train = all_train[np.arange(n_repetitions), selected_degrees]
selected_val = all_val[np.arange(n_repetitions), selected_degrees]
selected_test = all_test[np.arange(n_repetitions), selected_degrees]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(selected_degrees, bins=np.arange(degrees[-1] + 2) - 0.5,
             edgecolor="black")
axes[0].set(
    xlabel="Validation-selected degree",
    ylabel="Count",
    xticks=degrees[::2],
)
axes[1].boxplot(
    [selected_train, selected_val, selected_test],
    tick_labels=["Train", "Validation", "Test"],
)
axes[1].set(ylabel="MSE at the selected degree")
fig.tight_layout()
plt.show()

print(f"Mean selected validation MSE: {selected_val.mean():.3f}")
print(f"Mean corresponding test MSE: {selected_test.mean():.3f}")


Across repetitions, validation and test error curves are similar when evaluated degree by degree. At the **validation-selected** degree, validation error is systematically lower: selection found a model with both good performance and, often, favorable validation noise.


### Capacity of the Selection Procedure

The model family has capacity, but so does the search. For example, let $K$ be the maximum degree we are willing to consider and define
$$
\hat d_K=\arg\min_{0\le d\le K} e_d.
$$

The selected validation error
$$
e_{\hat d_K}=\min_{0\le d\le K}e_d
$$
can never increase with $K$: enlarging the search retains every old option. Test error has no such monotonicity guarantee.

This is a useful model of **meta-capacity**. A more extensive search can improve the selected model, but it also creates more opportunities to overfit validation noise.


In [ ]:
#| code-fold: true
selected_val_by_K = []
selected_test_by_K = []
selected_degree_by_K = []

for r in results:
    val_by_K = []
    test_by_K = []
    degree_by_K = []
    for K in degrees:
        best_degree = int(np.argmin(r["val_mse"][: K + 1]))
        degree_by_K.append(best_degree)
        val_by_K.append(r["val_mse"][best_degree])
        test_by_K.append(r["test_mse"][best_degree])
    selected_val_by_K.append(val_by_K)
    selected_test_by_K.append(test_by_K)
    selected_degree_by_K.append(degree_by_K)

selected_val_by_K = np.array(selected_val_by_K)
selected_test_by_K = np.array(selected_test_by_K)
selected_degree_by_K = np.array(selected_degree_by_K)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(degrees, np.median(selected_degree_by_K, axis=0))
axes[0].set(
    xlabel="Maximum degree considered, $K$",
    ylabel="Median selected degree",
    xticks=degrees[::2],
)

axes[1].plot(
    degrees,
    np.median(selected_val_by_K, axis=0),
    label="Selected validation MSE",
)
axes[1].plot(
    degrees,
    np.median(selected_test_by_K, axis=0),
    label="Corresponding test MSE",
)
axes[1].set(
    xlabel="Maximum degree considered, $K$",
    ylabel="Median MSE",
    xticks=degrees[::2],
)
axes[1].legend()
fig.tight_layout()
plt.show()


## Using CV for Selection

A separate validation set is simple, but leaves fewer observations for fitting. Inside the same meta-procedure, we can replace the single validation score with
$$
e_\lambda=\operatorname{CV}(\mathcal D_{\mathrm{train}},A_\lambda).
$$

We then select the smallest $e_\lambda$, refit the selected procedure on all available training data, and evaluate its result on untouched test data.

Here `train` means all data available for model building; CV creates its own internal training and validation folds. The test set remains outside the entire selection procedure.

Again, CV estimates performance first. Comparing its estimates is what performs selection.


## Evaluate the Entire Pipeline

The meta-procedure $\mathcal P$ may include preprocessing, feature selection, fitting several $A_\lambda$, comparing their CV estimates, selecting $\hat\lambda$, and refitting. An honest test evaluates the output of this **entire procedure**.

Anything learned from data must happen inside the training portion. For example, selecting features using all responses before splitting leaks evaluation information; splitting afterward does not undo it.

Pipelines help enforce this rule: fit scaling, imputation, polynomial features, or feature-selection steps separately inside each fold along with the estimator.

### Nested cross-validation

If we cross-validate the complete CV-based selection procedure $\mathcal P$, each outer training fold runs its own inner CV selection, and the outer evaluation fold assesses the result. This is **nested cross-validation**.

We will not develop its two loops here. CV-based selection followed by one evaluation on a separate test set illustrates the same essential separation.


## In Practice: Penguins

We now repeat the main workflows on the three-species penguins data from the previous lecture. Use flipper length and bill length to predict species, and evaluate predictions using accuracy.

The examples below demonstrate three different uses of the data. Treat each as a separate analysis rather than consecutive stages of one analysis.


In [ ]:
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.preprocessing import StandardScaler

url = "https://gist.githubusercontent.com/slopp/ce3b90b9168f2f921784de84fa445651/raw/penguins.csv"
#features = ["flipper_length_mm", "bill_length_mm"]

#penguins = pd.read_csv(url)[features + ["species"]].dropna()
penguins = pd.read_csv(url).dropna()
X_peng = penguins[features].to_numpy()
y_peng = penguins["species"].to_numpy()

print(f"Observations: {len(penguins)}")
print(penguins["species"].value_counts())


### A Train/Test Split

First evaluate one fixed procedure: standardize the two features, then fit multiclass logistic regression. Stratification keeps the species proportions similar in the two subsets.


In [ ]:
def penguin_classifier(degree=1):
    return make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        StandardScaler(),
        LogisticRegression(C=np.inf, max_iter=10000),
    )


X_train, X_test, y_train, y_test = train_test_split(
    X_peng,
    y_peng,
    test_size=0.20,
    stratify=y_peng,
    random_state=460,
)

linear_classifier = penguin_classifier(degree=1)
linear_classifier.fit(X_train, y_train)
test_predictions = linear_classifier.predict(X_test)

print(f"Training observations: {len(y_train)}")
print(f"Test observations: {len(y_test)}")
print(f"Test accuracy: {accuracy_score(y_test, test_predictions):.3f}")


In [ ]:
#| code-fold: true
ConfusionMatrixDisplay.from_predictions(y_test, test_predictions)
plt.title("Penguins: test-set predictions")
plt.show()


The test observations did not fit the scaler or the classifier. Test accuracy therefore estimates the performance of this particular fitted model on new penguins from the same population.

### Cross-Validation

Next hold the procedure fixed and estimate its performance using five-fold CV. Both standardization and logistic fitting occur separately within every training fold because they are in the pipeline.


In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=461)
cv_accuracy = cross_val_score(
    penguin_classifier(degree=1),
    X_peng,
    y_peng,
    cv=folds,
    scoring="accuracy",
)

print("Fold accuracies:", np.round(cv_accuracy, 3))
print(f"Mean CV accuracy: {cv_accuracy.mean():.3f}")


These five scores come from five fitted models. None is a final model. Their average estimates the performance of the fixed degree-1 fitting procedure.

### Train/Validation/Test Selection

Finally, use validation accuracy to choose polynomial degree $d$. If validation accuracy ties, choose the smaller degree.


In [ ]:
# First reserve the test set. Then split the remaining data into
# training and validation sets: 60% / 20% / 20% overall.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_peng,
    y_peng,
    test_size=0.20,
    stratify=y_peng,
    random_state=460,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    stratify=y_train_val,
    random_state=1460,
)

candidate_degrees = np.array([1, 5, 20])
validation_accuracy = []

for degree in candidate_degrees:
    model = penguin_classifier(degree)
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    validation_accuracy.append(accuracy_score(y_val, predictions))

validation_accuracy = np.array(validation_accuracy)
best_accuracy = validation_accuracy.max()
selected_degree = int(candidate_degrees[validation_accuracy == best_accuracy].min())

for degree, accuracy in zip(candidate_degrees, validation_accuracy):
    print(f"Degree {degree}: validation accuracy = {accuracy:.3f}")
print(f"Selected degree: {selected_degree}")


In [ ]:
# Refit the selected procedure on training plus validation data.
final_classifier = penguin_classifier(selected_degree)
final_classifier.fit(X_train_val, y_train_val)

# Use the test set only after selection and refitting are complete.
final_predictions = final_classifier.predict(X_test)
final_test_accuracy = accuracy_score(y_test, final_predictions)

print(f"Final test accuracy: {final_test_accuracy:.3f}")


The validation scores selected the degree, so the winning validation accuracy is part of model building. The final test accuracy evaluates the refitted output of the whole procedure:

1. create polynomial features,
2. learn the scaling transformation,
3. fit logistic-regression coefficients,
4. compare degrees using validation accuracy,
5. select a degree and refit.

This small example is the practical version of the meta-procedure $\mathcal P$.



## Review Questions

See: @sec-model-eval-selection-questions
